# Semantic abstraction graphicalizer

Thin experiment notebook: configure the run, invoke the package, and visualize the resulting graph, sample a connected subgraph, and narrate it.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import textwrap

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'assets').exists():
    raise FileNotFoundError('Could not locate the repository assets directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graphicalizer import (
    ExtractionDensityConfig,
    Graphicalizer,
    GraphicalizerConfig,
    NetworkXGraphStore,
    NodeEmbeddingConfig,
    NodeContextConfig,
    SubgraphNarrator,
    SubgraphNarrativeConfig,
    SubgraphNarrativePrompt,
    sample_random_connected_subgraph,
    graph_to_dot,
    load_ontology,
    load_corpus_articles,
    render_graph,
    validate_ontology_labeled_graph,
)
from IPython.display import SVG, display

## Configure and run

In [ ]:
from graphicalizer.notebook_config import (
    debug_pathogens,
    limit_debug_abstracts,
    load_notebook_config,
    resolve_config_path,
)

CONFIG = load_notebook_config(PROJECT_ROOT)
COMMON = CONFIG['common']
SETTINGS = CONFIG['notebook_05_semantic_abstraction']
DEBUG_MODE = COMMON['debug_mode']
DEBUG_PATHOGENS = debug_pathogens(COMMON)
ASSETS_ROOT = PROJECT_ROOT / 'assets'
CORPUS_PATH = resolve_config_path(
    PROJECT_ROOT,
    Path(COMMON['output_root'])
    / COMMON['corpus_subdir']
    / ('debug' if DEBUG_MODE else '')
    / COMMON['corpus_filename'],
)
CORPUS_PATHOGENS = (
    list(DEBUG_PATHOGENS)
    if DEBUG_MODE
    else COMMON['corpus_pathogens']
)
CORPUS_START_YEAR = COMMON['corpus_start_year']
CORPUS_END_YEAR = COMMON['corpus_end_year']
PUBMED_ABSTRACT_INDEX = SETTINGS['pubmed_abstract_index']
CORPUS_ARTICLES = load_corpus_articles(
    CORPUS_PATH,
    pathogens=CORPUS_PATHOGENS,
    start_year=CORPUS_START_YEAR,
    end_year=CORPUS_END_YEAR,
)
CORPUS_ARTICLES = limit_debug_abstracts(CORPUS_ARTICLES, COMMON)
if CORPUS_ARTICLES.empty:
    raise ValueError('No abstracts remain after corpus filtering.')
if not 0 <= PUBMED_ABSTRACT_INDEX < len(CORPUS_ARTICLES):
    raise IndexError(
        f'PUBMED_ABSTRACT_INDEX must be between 0 and {len(CORPUS_ARTICLES) - 1}.'
    )
ABSTRACT_ROW = CORPUS_ARTICLES.iloc[PUBMED_ABSTRACT_INDEX]
ABSTRACT_REF = f'{ABSTRACT_ROW["pathogen"]}/{ABSTRACT_ROW["pmid"]}'
ASSEMBLED_ONTOLOGY_PATH = resolve_config_path(PROJECT_ROOT, COMMON['assembled_ontology_relative_path'])
ONTOLOGY_PATH = (
    ASSEMBLED_ONTOLOGY_PATH
    if ASSEMBLED_ONTOLOGY_PATH.exists()
    else resolve_config_path(PROJECT_ROOT, COMMON['base_ontology_relative_path'])
)
PROMPT_PATH = resolve_config_path(PROJECT_ROOT, COMMON['prompt_relative_path'])
PROMPT_SNAPSHOT_PATH = resolve_config_path(PROJECT_ROOT, COMMON['prompt_snapshot_relative_path'])
GRAPH_OUTPUT_PATH = resolve_config_path(PROJECT_ROOT, SETTINGS['graph_output_relative_path'])
GRAPH_STORE_PATH = resolve_config_path(PROJECT_ROOT, COMMON['graph_store_subdir'])
LLM_PROVIDER = COMMON['llm_provider']
LLM_MODEL = COMMON['openai_model'] if LLM_PROVIDER == 'openai' else COMMON['ollama_model']
LLM_OPTIONS = (
    {'max_output_tokens': COMMON['openai_max_output_tokens']}
    if LLM_PROVIDER == 'openai'
    else {
        'num_ctx': COMMON['ollama_num_ctx'],
        'num_predict': COMMON['ollama_num_predict'],
    }
)
NARRATIVE_COLUMNS = SETTINGS['narrative_columns']
EMBEDDING_MODEL_NAME = COMMON['embedding_model_name']
from sentence_transformers import SentenceTransformer
EMBEDDING_MODEL = SentenceTransformer(EMBEDDING_MODEL_NAME)
EMBEDDING_CONFIG = NodeEmbeddingConfig(
    normalize=True,
    model_id=EMBEDDING_MODEL_NAME,
)

abstract_text = str(ABSTRACT_ROW['abstract'])
print('Selected corpus row:', ABSTRACT_REF)
print(textwrap.fill(abstract_text, width=NARRATIVE_COLUMNS))

ontology = load_ontology(ONTOLOGY_PATH)
density = ExtractionDensityConfig(
    entities_per_word=COMMON['entities_per_word'],
    relations_per_entity=COMMON['relations_per_entity'],
    minimum_entity_fraction=COMMON['minimum_entity_fraction'],
    density_retries=COMMON['density_retries'],
)
context = NodeContextConfig(
    max_sentences=COMMON['max_context_sentences'],
    max_evidence_items=COMMON['max_evidence_items'],
    include_evidence=COMMON['include_evidence'],
    include_uncertainty=COMMON['include_uncertainty'],
)
config = GraphicalizerConfig(
    provider=LLM_PROVIDER,
    model=LLM_MODEL,
    extraction_density=density,
    node_context=context,
    prompt_template_path=PROMPT_PATH,
    prompt_snapshot_path=PROMPT_SNAPSHOT_PATH,
    disconnected_policy=COMMON['disconnected_policy'],
    context_policy=COMMON['context_policy'],
    casting_retries=COMMON['casting_retries'],
)

graphicalizer = Graphicalizer.from_provider(
    ontology,
    config,
    options=LLM_OPTIONS,
    embedding_model=EMBEDDING_MODEL,
    embedding_config=EMBEDDING_CONFIG,
)
result = graphicalizer.run(abstract_text)


In [ ]:
print('Model:', config.model)
print('Target entities:', density.target_counts(abstract_text)['target_entity_count'])
print('Minimum entities:', density.target_counts(abstract_text)['minimum_entity_count'])
print('Source SHA-256:', result.run_metadata['source_sha256'])
print('Raw validation:', result.raw_validation.to_dict())
print('Normalized validation:', result.normalized_validation.to_dict())
print('Normalization report:', result.normalization_report)
print('Final validation:', result.final_validation.to_dict())
print('Entities:', len(result.normalized_extraction.entities))
print('Relations:', len(result.ontology_casting.relations))
print('Node contexts:', len(result.node_contexts))
print('Embedding metadata:', result.graph.graph.get('node_context_embeddings'))

In [ ]:
GRAPH_STORE = NetworkXGraphStore(GRAPH_STORE_PATH)
GRAPH_ID = f'{COMMON["graph_id_prefix"]}_{ABSTRACT_ROW["pathogen"]}_{ABSTRACT_ROW["pmid"]}'
GRAPH_PATH = GRAPH_STORE.save(result.graph, GRAPH_ID)
LOADED_GRAPH = GRAPH_STORE.load(GRAPH_ID)
print('Saved graph:', GRAPH_PATH)
print('Stored graphs:', len(GRAPH_STORE.list()))
print('Loaded nodes:', LOADED_GRAPH.number_of_nodes())
print('Loaded edges:', LOADED_GRAPH.number_of_edges())


## Visualize the typed graph

In [ ]:
validate_ontology_labeled_graph(result.typed_graph)
dot_text = graph_to_dot(
    result.typed_graph,
    graph_name='SelectedPubMedGraph',
    require_ontology_labels=True,
)
print(dot_text)
render_graph(
    result.typed_graph,
    GRAPH_OUTPUT_PATH,
    graph_name='SelectedPubMedGraph',
    require_ontology_labels=True,
)
display(SVG(filename=str(GRAPH_OUTPUT_PATH)))

## Sample and narrate a connected subgraph

In [ ]:
SUBGRAPH_NODE_COUNT = SETTINGS['subgraph_node_count']
SUBGRAPH_SEED = SETTINGS['subgraph_seed']
NARRATIVE_WORDS = SETTINGS['narrative_words']
SUBGRAPH_OUTPUT_PATH = resolve_config_path(PROJECT_ROOT, SETTINGS['subgraph_output_relative_path'])
NARRATIVE_PROMPT_PATH = resolve_config_path(PROJECT_ROOT, SETTINGS['narrative_prompt_relative_path'])
NARRATIVE_SNAPSHOT_PATH = resolve_config_path(PROJECT_ROOT, SETTINGS['narrative_snapshot_relative_path'])

sampled_subgraph = sample_random_connected_subgraph(
    result.typed_graph,
    num_nodes=SUBGRAPH_NODE_COUNT,
    seed=SUBGRAPH_SEED,
)
validate_ontology_labeled_graph(sampled_subgraph)
render_graph(
    sampled_subgraph,
    SUBGRAPH_OUTPUT_PATH,
    graph_name='SelectedRandomSubgraph',
    require_ontology_labels=True,
)
display(SVG(filename=str(SUBGRAPH_OUTPUT_PATH)))


In [ ]:

narrator_prompt = SubgraphNarrativePrompt.from_yaml_file(NARRATIVE_PROMPT_PATH)
narrator = SubgraphNarrator.from_graphicalizer(
    graphicalizer,
    prompt=narrator_prompt,
    prompt_snapshot_path=NARRATIVE_SNAPSHOT_PATH,
)
narrative = narrator.narrate(
    sampled_subgraph,
    SubgraphNarrativeConfig(target_words=NARRATIVE_WORDS),
)
print(textwrap.fill(narrative.narrative, width=NARRATIVE_COLUMNS))
print('Requested words:', narrative.requested_words)
print('Actual words:', narrative.actual_words)
if narrative.uncertainty:
    print('Uncertainty:', narrative.uncertainty)


## Inspect node contexts

In [ ]:
for node_context in result.node_contexts:
    print(f'\n[{node_context.entity_id}] {textwrap.fill(node_context.summary, width=NARRATIVE_COLUMNS)}')
    for evidence in node_context.evidence:
        print(f'  evidence: {textwrap.fill(evidence, width=NARRATIVE_COLUMNS)}')
    if node_context.uncertainty:
        print(f'  uncertainty: {textwrap.fill(node_context.uncertainty, width=NARRATIVE_COLUMNS)}')